# 04 · Gold analytics

Turn the clean silver tables into decision-ready gold. First the core
`item_performance` table — one row per item with a **conversion rate**
(`orders / pageviews`), the payoff of the medallion pipeline. Then **stage 5**:
four more gold tables (top sellers, hourly sales, traffic by channel, RFM-style
user engagement segments) plus a CSV export to object storage — mirroring
`50_gold_analytics.py` so this notebook reproduces the full `make pipeline` gold layer.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("04-gold-analytics").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| default catalog:", spark.conf.get("spark.sql.defaultCatalog"))

In [ ]:
from pyspark.sql.functions import coalesce, col, count, lit, sum, when

items = spark.table("demo.silver.items")
purchases = spark.table("demo.silver.purchases_enriched")
pageviews = spark.table("demo.silver.pageviews_by_items")

purchase_agg = purchases.groupBy("item_id").agg(
    sum("quantity").alias("items_sold"),
    count("id").alias("orders"),
    sum("total_price").alias("revenue"),
)
pageview_agg = (
    pageviews.filter(col("page") == "products")
    .groupBy("item_id").agg(count(lit(1)).alias("pageviews"))
)

gold = (
    items.select(col("id").alias("item_id"), col("name").alias("item_name"),
                 col("category").alias("item_category"))
    .join(purchase_agg, "item_id", "left")
    .join(pageview_agg, "item_id", "left")
    .select(
        "item_id", "item_name", "item_category",
        coalesce(col("items_sold"), lit(0)).cast("long").alias("items_sold"),
        coalesce(col("orders"), lit(0)).cast("long").alias("orders"),
        coalesce(col("revenue"), lit(0)).cast("decimal(20,2)").alias("revenue"),
        coalesce(col("pageviews"), lit(0)).cast("long").alias("pageviews"),
        when(coalesce(col("pageviews"), lit(0)) > 0, col("orders") / col("pageviews"))
            .otherwise(lit(0.0)).cast("double").alias("conversion_rate"),
    )
)
gold.write.format("iceberg").mode("overwrite").save("demo.gold.item_performance")
print("demo.gold.item_performance:", gold.count(), "rows")

## Top items by revenue

In [ ]:
%%sql
SELECT item_name, item_category, orders, revenue, pageviews,
       round(conversion_rate, 4) AS conversion_rate
FROM demo.gold.item_performance
ORDER BY revenue DESC
LIMIT 10

## Best converting items (with real traffic)

In [ ]:
%%sql
SELECT item_name, item_category, orders, pageviews,
       round(conversion_rate, 4) AS conversion_rate
FROM demo.gold.item_performance
WHERE pageviews > 0
ORDER BY conversion_rate DESC
LIMIT 10

## Revenue by category

In [ ]:
%%sql
SELECT item_category,
       sum(orders)   AS orders,
       sum(revenue)  AS revenue,
       sum(pageviews) AS pageviews
FROM demo.gold.item_performance
GROUP BY item_category
ORDER BY revenue DESC

## Extended gold analytics (stage 5)

Stage 4 above produced `item_performance`. The batch pipeline's **stage 5**
(`docker/spark/pipeline/50_gold_analytics.py`, run automatically by `make pipeline`)
builds four more gold tables and exports one of them as CSV. The cells below
reproduce that stage interactively, so the notebook path matches `make pipeline`:

| Table | Question it answers |
|---|---|
| `top_selling_items` | Which 10 products earn the most revenue? |
| `sales_performance_24h` | How did hourly revenue look over the last 24 h? |
| `pageviews_by_channel` | Which channels drive product-page traffic? |
| `user_engagement_segments` | Who are the high / medium / low engagement users? |

Plus a CSV export of the engagement segments to `s3a://customer-segments/` — the
**data-product delivery** pattern, handing gold data to a consumer that doesn't
speak Iceberg.

In [ ]:
from pyspark.sql.functions import (
    countDistinct, current_date, current_timestamp, datediff, expr, max, to_date,
)

# 1. top_selling_items — top 10 by revenue (materialised so a dashboard needn't re-aggregate)
top_items = (
    spark.table("demo.silver.purchases_enriched")
    .groupBy("item_id", "item_name", "item_category")
    .agg(sum("total_price").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
    .limit(10)
)
top_items.write.format("iceberg").mode("overwrite").save("demo.gold.top_selling_items")
print("demo.gold.top_selling_items:", top_items.count(), "rows")

# 2. sales_performance_24h — hourly revenue in the last 24h. Static snapshot, not a
# streaming window: the loadgen scatters purchases across ~24h, so the first run populates it.
perf_24h = (
    spark.table("demo.silver.purchases_enriched")
    .filter(col("created_at") >= current_timestamp() - expr("INTERVAL 24 HOURS"))
    .groupBy("purchase_hour")
    .agg(sum("total_price").alias("total_revenue"))
    .orderBy("purchase_hour")
)
perf_24h.write.format("iceberg").mode("overwrite").save("demo.gold.sales_performance_24h")
print("demo.gold.sales_performance_24h:", perf_24h.count(), "rows")

# 3. pageviews_by_channel — product-page traffic per channel
pvs_by_channel = (
    spark.table("demo.silver.pageviews_by_items")
    .groupBy("channel")
    .agg(count(lit(1)).alias("total_pageviews"))
    .orderBy(col("total_pageviews").desc())
)
pvs_by_channel.write.format("iceberg").mode("overwrite").save("demo.gold.pageviews_by_channel")
print("demo.gold.pageviews_by_channel:", pvs_by_channel.count(), "rows")

In [ ]:
%%sql
SELECT item_name, item_category, total_revenue
FROM demo.gold.top_selling_items
ORDER BY total_revenue DESC

In [ ]:
%%sql
SELECT channel, total_pageviews
FROM demo.gold.pageviews_by_channel
ORDER BY total_pageviews DESC

## User engagement segments (RFM-style)

**RFM** = **R**ecency, **F**requency, **M**onetary — a classic marketing model that
scores customers on how *recently* they engaged, how *often*, and how *much* they
spend. This table implements an **RF** variant: recency (`days_since_last_active`)
and frequency (`total_pageviews`) only, **omitting the monetary axis** to stay
focused on the pageview stream. (For full RFM, join a per-user `sum(total_price)`
from `silver.purchases_enriched` and add it as a third scoring dimension.)

Only users with `valid_email = TRUE` are scored — a customer we can't reach isn't a
campaign target. The thresholds below are calibrated to this loadgen (1,000 users,
~6 pageviews each); re-tune them if you change `PAGEVIEWS_PER_PURCHASE`.

```
total_pageviews ≥ 8  AND  days_since_last_active ≤ 3  →  high_engagement
total_pageviews ≥ 3  AND  days_since_last_active ≤ 7  →  medium_engagement
otherwise                                             →  low_engagement
```

In [ ]:
# silver.users keys on `id`; only pageviews_by_items carries `user_id` — alias first,
# otherwise the join / groupBy on "user_id" fail to resolve.
users = spark.table("demo.silver.users").select(
    col("id").alias("user_id"), "email", "full_name", "valid_email"
)
pvs = spark.table("demo.silver.pageviews_by_items")

segments = (
    users.join(pvs, "user_id", "left")
    .filter(col("valid_email") == True)          # only reachable users are scored
    .groupBy("user_id", "email", "full_name")
    .agg(
        count("page").alias("total_pageviews"),                        # frequency
        countDistinct(to_date(col("received_at"))).alias("active_days"),
        max(to_date(col("received_at"))).alias("last_active_date"),
    )
    .withColumn(                                                       # recency
        "days_since_last_active", datediff(current_date(), col("last_active_date"))
    )
    .withColumn(
        "engagement_segment",
        when((col("total_pageviews") >= 8) & (col("days_since_last_active") <= 3),
             lit("high_engagement"))
        .when((col("total_pageviews") >= 3) & (col("days_since_last_active") <= 7),
              lit("medium_engagement"))
        .otherwise(lit("low_engagement")),
    )
)
segments.write.format("iceberg").mode("overwrite").save("demo.gold.user_engagement_segments")
print("demo.gold.user_engagement_segments:", segments.count(), "rows")

In [ ]:
%%sql
SELECT engagement_segment, count(*) AS user_count
FROM demo.gold.user_engagement_segments
GROUP BY engagement_segment
ORDER BY user_count DESC

In [ ]:
# Data-product delivery: push the segments out as a single CSV to object storage
# so a non-lakehouse consumer (marketing / CRM / spreadsheet) can pick it up.
# s3a:// creds come from spark-defaults.conf in the image — no AWS_* env vars needed.
segments.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "s3a://customer-segments/segmented_users"
)
print("CSV exported to s3a://customer-segments/segmented_users/")
print("Spark writes part-00000-<uuid>.csv (+ a _SUCCESS marker); consumers glob on *.csv")